<a href="https://colab.research.google.com/github/avithpcs24-coder/BIS/blob/main/Particleswarm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np

# ==========================================
# 1. NETWORK TRAFFIC DATA
# ==========================================

# Traffic scores of normal users
normal_traffic = np.array([
    20, 25, 30, 28, 35, 32, 27, 24, 31, 29,
    34, 26, 33, 30, 36
])

# Traffic scores when an intruder is present
intrusion_traffic = np.array([
    70, 75, 82, 90, 85, 95, 88, 80, 92, 98
])


# ==========================================
# 2. DETECTION FUNCTION
# ==========================================

def detect(traffic, threshold):

    if traffic >= threshold:
        return 1       # Intrusion

    return 0           # Normal


# ==========================================
# 3. FITNESS FUNCTION
# ==========================================

def fitness(threshold):

    false_alarms = 0
    detected_attacks = 0

    # Check normal traffic
    for traffic in normal_traffic:

        if detect(traffic, threshold) == 1:
            false_alarms += 1

    # Check intrusion traffic
    for traffic in intrusion_traffic:

        if detect(traffic, threshold) == 1:
            detected_attacks += 1

    detection_rate = (
        detected_attacks / len(intrusion_traffic)
    )

    false_alarm_rate = (
        false_alarms / len(normal_traffic)
    )

    # Lower fitness is better
    #
    # We strongly penalize missed attacks
    # and false alarms.

    score = (
        5 * (1 - detection_rate)
        +
        3 * false_alarm_rate
    )

    return score, detection_rate, false_alarm_rate


# ==========================================
# 4. PARTICLE SWARM OPTIMIZATION
# ==========================================

number_of_particles = 10
iterations = 30

# Each particle represents a possible threshold
particles = np.random.uniform(
    20, 90,
    number_of_particles
)

velocities = np.zeros(number_of_particles)

# Personal best
personal_best = particles.copy()
personal_best_score = np.array([
    fitness(x)[0] for x in particles
])

# Global best
best_index = np.argmin(personal_best_score)

global_best = personal_best[best_index]
global_best_score = personal_best_score[best_index]


# PSO constants
w = 0.7
c1 = 1.5
c2 = 1.5


# ==========================================
# 5. RUN PSO
# ==========================================

for iteration in range(iterations):

    for i in range(number_of_particles):

        r1 = np.random.random()
        r2 = np.random.random()

        # Update velocity
        velocities[i] = (
            w * velocities[i]
            +
            c1 * r1 *
            (personal_best[i] - particles[i])
            +
            c2 * r2 *
            (global_best - particles[i])
        )

        # Update position
        particles[i] += velocities[i]

        # Keep threshold between 20 and 90
        particles[i] = np.clip(
            particles[i],
            20,
            90
        )

        # Calculate new fitness
        current_score = fitness(
            particles[i]
        )[0]

        # Update personal best
        if current_score < personal_best_score[i]:

            personal_best[i] = particles[i]
            personal_best_score[i] = current_score

        # Update global best
        if current_score < global_best_score:

            global_best = particles[i]
            global_best_score = current_score


# ==========================================
# 6. FINAL OPTIMAL THRESHOLD
# ==========================================

optimal_threshold = global_best

score, detection_rate, false_alarm_rate = fitness(
    optimal_threshold
)


print("\n======================================")
print("       PSO OPTIMIZATION RESULT")
print("======================================")

print(
    "Optimal Threshold :",
    round(optimal_threshold, 2)
)

print(
    "Detection Rate    :",
    round(detection_rate * 100, 2),
    "%"
)

print(
    "False Alarm Rate  :",
    round(false_alarm_rate * 100, 2),
    "%"
)


# ==========================================
# 7. TEST NORMAL TRAFFIC
# ==========================================

print("\n======================================")
print("          NORMAL TRAFFIC")
print("======================================")

for traffic in normal_traffic:

    result = detect(
        traffic,
        optimal_threshold
    )

    if result == 1:

        print(
            "Traffic =", traffic,
            "-> FALSE ALARM"
        )

    else:

        print(
            "Traffic =", traffic,
            "-> NORMAL"
        )


# ==========================================
# 8. TEST INTRUSION TRAFFIC
# ==========================================

print("\n======================================")
print("          INTRUSION TRAFFIC")
print("======================================")

for traffic in intrusion_traffic:

    result = detect(
        traffic,
        optimal_threshold
    )

    if result == 1:

        print(
            "Traffic =", traffic,
            "-> INTRUSION DETECTED"
        )

    else:

        print(
            "Traffic =", traffic,
            "-> MISSED INTRUSION"
        )


       PSO OPTIMIZATION RESULT
Optimal Threshold : 69.56
Detection Rate    : 100.0 %
False Alarm Rate  : 0.0 %

          NORMAL TRAFFIC
Traffic = 20 -> NORMAL
Traffic = 25 -> NORMAL
Traffic = 30 -> NORMAL
Traffic = 28 -> NORMAL
Traffic = 35 -> NORMAL
Traffic = 32 -> NORMAL
Traffic = 27 -> NORMAL
Traffic = 24 -> NORMAL
Traffic = 31 -> NORMAL
Traffic = 29 -> NORMAL
Traffic = 34 -> NORMAL
Traffic = 26 -> NORMAL
Traffic = 33 -> NORMAL
Traffic = 30 -> NORMAL
Traffic = 36 -> NORMAL

          INTRUSION TRAFFIC
Traffic = 70 -> INTRUSION DETECTED
Traffic = 75 -> INTRUSION DETECTED
Traffic = 82 -> INTRUSION DETECTED
Traffic = 90 -> INTRUSION DETECTED
Traffic = 85 -> INTRUSION DETECTED
Traffic = 95 -> INTRUSION DETECTED
Traffic = 88 -> INTRUSION DETECTED
Traffic = 80 -> INTRUSION DETECTED
Traffic = 92 -> INTRUSION DETECTED
Traffic = 98 -> INTRUSION DETECTED
